# How has Gender Shifted Culturally Over Time?


The goal of this notebook is to explore the role that *AUTHOR GENDER* plays in the combined **NYT Best Sellers** and **Goodreads Top Ranked Novels** dataset. I will start by exploring what what genders dominate throughout time in the top 500 dataset, and whether the ranking for each gender changes throughout time as well. Then I'll move over to the merged dataset, where I can discuss novels that received a high ranking AND high praise, and whether they are more often male or female.

### Imports

This cell imports the necessary libraries for all of the EDA practicies throughout the notebook, as well as load the dataset that I'll be using. 

In [220]:
import pandas as pd
import numpy as np
import altair as alt

# Load Top 500 Novels dataset (tab-separated)
df_top = pd.read_csv("https://raw.githubusercontent.com/melaniewalsh/responsible-datasets-in-context/main/datasets/top-500-novels/final_merged_dataset_no_full_text.tsv",
                        sep="\t", encoding="utf-8")

# Load NYT Bestsellers dataset
df_nyt = pd.read_csv("https://raw.githubusercontent.com/ecds/post45-datasets/main/nyt_full.tsv",sep="\t",encoding="utf-8")

### Initial Exploration

In [221]:
print("Top 500 Novels dataset\n")
df_top.info()


Top 500 Novels dataset

<class 'pandas.DataFrame'>
RangeIndex: 500 entries, 0 to 499
Data columns (total 29 columns):
 #   Column                    Non-Null Count  Dtype  
---  ------                    --------------  -----  
 0   top_500_rank              500 non-null    int64  
 1   title                     500 non-null    str    
 2   author                    500 non-null    str    
 3   pub_year                  500 non-null    int64  
 4   orig_lang                 500 non-null    str    
 5   genre                     500 non-null    str    
 6   author_birth              499 non-null    str    
 7   author_death              496 non-null    str    
 8   author_gender             500 non-null    str    
 9   author_primary_lang       500 non-null    str    
 10  author_nationality        500 non-null    str    
 11  author_field_of_activity  329 non-null    str    
 12  author_occupation         459 non-null    str    
 13  oclc_holdings             495 non-null    float64
 1

Focusing on columns that are going to be relied on heavily for our analysis, I can see that *author_gender* is not missing any data, as well as *top_500_rank* and *pub_year*. While there are some rows with missing data, they appear to be very specific rows, and not general eda focused variables. 

The same columns also have expected data types, though *pub_year* will need to be converted to a datetime format. 

In [222]:
print("NYT Bestsellers dataset\n")
df_nyt.info()

NYT Bestsellers dataset

<class 'pandas.DataFrame'>
RangeIndex: 60386 entries, 0 to 60385
Data columns (total 6 columns):
 #   Column    Non-Null Count  Dtype
---  ------    --------------  -----
 0   year      60386 non-null  int64
 1   week      60386 non-null  str  
 2   rank      60386 non-null  int64
 3   title_id  60386 non-null  int64
 4   title     60386 non-null  str  
 5   author    60376 non-null  str  
dtypes: int64(3), str(3)
memory usage: 2.8 MB


The NYT dataset doesn't have as many columns, but there are lots of rows available for analysis. It has very little missing data, with only ten rows missing form the author column, out of 60386 rows. 

In [223]:
display(df_top.head())
display(df_nyt.head())

,top_500_rank,title,author,pub_year,orig_lang,genre,author_birth,author_death,author_gender,author_primary_lang,...,gr_num_ratings,gr_num_reviews,gr_avg_rating_rank,gr_num_ratings_rank,oclc_owi,author_viaf,gr_url,wiki_url,pg_eng_url,pg_orig_url
0,1,Don Quixote,Miguel de Cervantes,1605,Spanish,action,1547,1616,male,spa,...,"269,435","12,053",318,211,1.810748e+09,17220427,https://www.goodreads.com/book/show/3836.Don_Q...,https://en.wikipedia.org/wiki/Don_Quixote,https://www.gutenberg.org/cache/epub/996/pg996...,https://www.gutenberg.org/cache/epub/2000/pg20...
1,2,Alice's Adventures in Wonderland,Lewis Carroll,1865,English,fantasy,1832,1898,male,eng,...,"561,016","15,380",172,133,1.156132e+10,66462036,https://www.goodreads.com/book/show/24213.Alic...,https://en.wikipedia.org/wiki/Alice%27s_Advent...,https://www.gutenberg.org/cache/epub/11/pg11.txt,NaN
2,3,The Adventures of Huckleberry Finn,Mark Twain,1884,English,action,1835,1910,male,eng,...,"1,262,480","19,440",373,68,3.373178e+09,50566653,https://www.goodreads.com/book/show/2956.The_A...,https://en.wikipedia.org/wiki/Adventures_of_Hu...,https://www.gutenberg.org/cache/epub/76/pg76.txt,NaN
3,4,The Adventures of Tom Sawyer,Mark Twain,1876,English,action,1835,1910,male,eng,...,"931,898","13,603",301,88,3.373178e+09,50566653,https://www.goodreads.com/book/show/24583.The_...,https://en.wikipedia.org/wiki/The_Adventures_o...,https://www.gutenberg.org/cache/epub/74/pg74.txt,NaN
4,5,Treasure Island,Robert Louis Stevenson,1883,English,action,1850,1894,male,eng,...,"486,155","16,307",368,145,3.434000e+03,95207986,https://www.goodreads.com/book/show/295.Treasu...,https://en.wikipedia.org/wiki/Treasure_Island,https://www.gutenberg.org/cache/epub/120/pg120...,NaN


,year,week,rank,title_id,title,author
0,1931,1931-10-12,1,6477,THE TEN COMMANDMENTS,Warwick Deeping
1,1931,1931-10-12,2,1808,FINCHE'S FORTUNE,Mazo de la Roche
2,1931,1931-10-12,3,5304,THE GOOD EARTH,Pearl S. Buck
3,1931,1931-10-12,4,4038,SHADOWS ON THE ROCK,Willa Cather
4,1931,1931-10-12,5,3946,SCARMOUCHE THE KING MAKER,Rafael Sabatini


These previews show the first few rows of each dataset so the column meanings and record structure are easier to inspect.

From the shape of the data we got in the *.info()* section, we can see how different these datasets present theselves. Majority of the useful data will come from the Goodreds dataset, with additional information being merged in from the NYT dataset. 

#### Initial Inspection: Gender

Now that I know the shape and fullness of the dataset, I want to know more about the current significant columns. These will be *author_gender*, *pub_year*, *author*, etc. 

In [224]:
print(f"There are {df_top['author'].nunique()} unique authors in the top 500 dataset")
print(f"There are {df_nyt['author'].nunique()} unique authors in the NYT dataset\n")

print(f'The top 500 dataset spans from {df_top["pub_year"].min()} to {df_top["pub_year"].max()}')
print(f'The NYT dataset spans from {df_nyt["year"].min()} to {df_nyt["year"].max()}\n')

print(f'There are {df_top[df_top["author_gender"] == "female"].shape[0]} books by female authors in the top 500 dataset')
print(f'There are {df_top[df_top["author_gender"] == "male"].shape[0]} books by male authors in the top 500 dataset')

There are 279 unique authors in the top 500 dataset
There are 2210 unique authors in the NYT dataset

The top 500 dataset spans from 1021 to 2015
The NYT dataset spans from 1931 to 2020

There are 145 books by female authors in the top 500 dataset
There are 355 books by male authors in the top 500 dataset


From the amount of authors in the NYT compared to top 500 dataset, I know there will be a significant amount of authors not represented in the merged dataset, but that can't be helped since gender is such an important detail. Only 279 actors will be represented. 

The top 500 dataset has a much longer span of time represented, but is missing more recent years than the NYT dataset. This could mean there is more data avilable from the top 500, but if majority of the data lays in the recent years, than the NYT has much more recent data.

There's also important information to see regarding the split between male and female authored novels. Male authors take up around 70% of the novels in the dataset, meaning we can expect the results to be male dominated at first, but there may be a pattern or trend regarding the female authors that we can't see yet.



### Merging the Datasets

The next step would be to merge the datasets together. I'll be merging left on the top 500 dataset, as to keep all of the data regarding the author's gender. This will cause the NYT data to be joined to the top 500 dataset. Inevitably, some of the NYT data will be lost, as not all of the authors/titles are present in both datasets. 

In [225]:
df_nyt = df_nyt.rename(columns={'title': 'nyt_title'})
df_nyt['title'] = df_nyt['nyt_title'].astype(str).str.strip().str.capitalize()

df_merged = df_top.merge(df_nyt, how='left', on=['author', 'title'])
df_merged.info()

<class 'pandas.DataFrame'>
RangeIndex: 721 entries, 0 to 720
Data columns (total 34 columns):
 #   Column                    Non-Null Count  Dtype  
---  ------                    --------------  -----  
 0   top_500_rank              721 non-null    int64  
 1   title                     721 non-null    str    
 2   author                    721 non-null    str    
 3   pub_year                  721 non-null    int64  
 4   orig_lang                 721 non-null    str    
 5   genre                     721 non-null    str    
 6   author_birth              720 non-null    str    
 7   author_death              717 non-null    str    
 8   author_gender             721 non-null    str    
 9   author_primary_lang       721 non-null    str    
 10  author_nationality        721 non-null    str    
 11  author_field_of_activity  519 non-null    str    
 12  author_occupation         680 non-null    str    
 13  oclc_holdings             716 non-null    float64
 14  oclc_eholdings       

The new merged dataset has 721 rows, with mostly non-null data!

### Gender-Focused EDA: Top 500 Novels

This section refocuses the analysis on author gender in the Top 500 novels dataset. I'm able to answer the following questions by using only the top 500 dataset:

- How does the proportion of male and female authors change by publication decade?
- Does the average Top 500 rank for female authors improve or worsen across time?

The merged NYT data is handled later because NYT weekly rows can duplicate the same book and would bias proportions or averages if used directly.

#### Clean Author Gender and Publication Year

This cleaning step creates a working dataframe called `df`. It keeps the analysis focused on books with usable author gender and publication year values, then creates a decade column so trends can be compared across time periods.

In [226]:
df = df_top.copy()

# Clean the gender and year columns
df = df.dropna(subset=["author_gender", "pub_year"]).copy()
df["pub_year"] = pd.to_numeric(df["pub_year"], errors="coerce")
df = df.dropna(subset=["pub_year"]).copy()
df["pub_year"] = df["pub_year"].astype(int)
# switch to decades for easier visualization
df["decade"] = (df["pub_year"] // 10) * 10

# Make rank numeric 
df["top_500_rank"] = pd.to_numeric(df["top_500_rank"], errors="coerce")

df[["title", "author", "author_gender", "pub_year", "decade", "top_500_rank"]].head()

,title,author,author_gender,pub_year,decade,top_500_rank
0,Don Quixote,Miguel de Cervantes,male,1605,1600,1
1,Alice's Adventures in Wonderland,Lewis Carroll,male,1865,1860,2
2,The Adventures of Huckleberry Finn,Mark Twain,male,1884,1880,3
3,The Adventures of Tom Sawyer,Mark Twain,male,1876,1870,4
4,Treasure Island,Robert Louis Stevenson,male,1883,1880,5


This shows the top 5 ranking books in the top_500 dataset. It may not look different, but we confirmed the datatypes for the variables we'll be working with.

#### Summary Tables

These tables support the main visualizations. *gender_counts* and *counts* show the number of Top 500 books by decade and gender, *gender_props* converts those counts into within-decade proportions, and *avg_rank* shows the mean Top 500 rank by decade and gender.

In [227]:
gender_counts = (
    df.groupby(["decade", "author_gender"])
    .size()
    .reset_index(name="count")
).sort_values(by=["decade"], ascending=False)

gender_props = gender_counts.copy()
gender_props["proportion"] = gender_props.groupby("decade")["count"].transform(lambda counts: counts / counts.sum())

avg_rank = (
    df.groupby(["decade", "author_gender"], as_index=False)
    .agg(average_rank=("top_500_rank", "mean"), books=("title", "count"))
).sort_values(by=["decade"], ascending=False)

counts = gender_counts.copy()

display(gender_counts.head())
display(gender_props.head())
display(avg_rank.head())

,decade,author_gender,count
53,2010,male,6
52,2010,female,5
51,2000,male,49
50,2000,female,25
49,1990,male,27


,decade,author_gender,count,proportion
53,2010,male,6,0.545455
52,2010,female,5,0.454545
51,2000,male,49,0.662162
50,2000,female,25,0.337838
49,1990,male,27,0.729730


,decade,author_gender,average_rank,books
53,2010,male,343.333333,6
52,2010,female,398.400000,5
51,2000,male,354.020408,49
50,2000,female,314.520000,25
49,1990,male,350.111111,27


These are only snippets of the data so we can't make full assumptions. We can better understand the format of the data, though. Each decade has a male and female row, which holds different statistics like count, proportion, etc. It seems like males tend to dominate the actual number of books, but the ranking switches back and forth between male and female. 

#### Gender Representation Over Time

This chart shows the share of Top 500 books in each publication decade by author gender. I am looking for whether female authors become a larger share of the dataset in more recent decades, and whether male authors dominate more strongly in earlier decades.

In [228]:
representation_chart = (
    alt.Chart(gender_props)
    .mark_line(point=True)
    .encode(
        x=alt.X("decade:O", title="Publication decade"),
        y=alt.Y("proportion:Q", title="Share of Top 500 books", axis=alt.Axis(format="%")),
        color=alt.Color("author_gender:N", title="Author gender"),
        tooltip=[
            alt.Tooltip("decade:O", title="Decade"),
            alt.Tooltip("author_gender:N", title="Author gender"),
            alt.Tooltip("count:Q", title="Books"),
            alt.Tooltip("proportion:Q", title="Share", format=".1%"),
        ],
    )
    .properties(
        title="Gender Representation in the Top 500 Novels Over Time",
        width=700,
        height=380,
    )
)

representation_chart

alt.Chart(...)

The main pattern to look for here is whether the lines move closer together over time. The plot starts out with very unbalanced data, as the male authors tend to have all of the books in really early recording. Around 1840, female authors start to become present in the data. As time goes on, the graph varies from decade to decade, but there is definetly an overall increase in the number of female authors in the top 500. By the last year, the male/female split is only 405% off from a perfect 50-50. This graph shows that more women authored novels started breaking into the top 500 as time went on.

#### Ranking Trends Over Time

This chart compares the average *top_500_rank* by decade and author gender. A lower rank means a book is placed higher on the Top 500 list, so a downward trend indicates improvement in average ranking.

In [229]:
rank_chart = (
    alt.Chart(avg_rank)
    .mark_line(point=True)
    .encode(
        x=alt.X("decade:O", title="Publication decade"),
        y=alt.Y("average_rank:Q", title="Average Top 500 rank", scale=alt.Scale(zero=False)),
        color=alt.Color("author_gender:N", title="Author gender"),
        tooltip=[
            alt.Tooltip("decade:O", title="Decade"),
            alt.Tooltip("author_gender:N", title="Author gender"),
            alt.Tooltip("average_rank:Q", title="Average rank", format=".1f"),
            alt.Tooltip("books:Q", title="Books"),
        ],
    )
    .properties(
        title="Average Top 500 Rank by Author Gender and Decade",
        width=700,
        height=380,
    )
)

rank_chart

alt.Chart(...)

For this ranking chart, I would compare both the direction of each line and the distance between gender groups. The earlier years are again very swayed because there was a low population of women authors present in the dataset. Surprisingly, the male and female plot looks pretty similar from 1830-2010. They both ironically start dropping in rank, as in the plot points get higher. This suggests not only do author rankings get worse over time, but the gender of the author is likely not the identifying variable here. The era of the novel and when it was written seems to have much more control over how well a book ranks. Many of the books authored before 1830 scores very high, hosting books with an average all better than top 50. Only 8 books written by women were represented in teh data after 1940.Overall, ranks seem to get worse as time goes on, for both men and women.

#### Contextual Count Plot

Counts help show how much data supports each decade-gender comparison. If a decade has only a few books for one gender group, then the proportion and average rank for that group should be interpreted carefully.

In [230]:
count_chart = (
    alt.Chart(counts)
    .mark_bar()
    .encode(
        x=alt.X("decade:O", title="Publication decade"),
        y=alt.Y("count:Q", title="Number of Top 500 books"),
        color=alt.Color("author_gender:N", title="Author gender"),
        tooltip=[
            alt.Tooltip("decade:O", title="Decade"),
            alt.Tooltip("author_gender:N", title="Author gender"),
            alt.Tooltip("count:Q", title="Books"),
        ],
    )
    .properties(
        title="Number of Top 500 Books by Gender and Decade",
        width=700,
        height=380,
    )
)

count_chart

alt.Chart(...)

This chart is mainly a check on the strength of the previous patterns. The early decades have nearly no books, barely 1-2 per decade. There is an occasional women author scatteree in the very early data. Starting in the early 1800s, the number of books start to pick up, revealing a clear upwards trend in volume. This means that as time went on, it was more commen for a book to be written and considered in the top 500. At the same time, the number of women in the top 500 group started growing consistently. Again, the value varied from decade to decade, but the overall rate of women authors getting their book into the top 500 seemed to keep growing. This graph shows that while time goes on, not only is it more liklely for authors to get their book in the top 500, but more likely for women authors too as well. 

### Merged Data: Currated Picks vs Bestseller Success

This optional section the currated picks, represented by the Top 500 list, with popularity or commercial success, represented by NYT bestseller appearances. Because the merged dataset can contain multiple weekly NYT rows for the same book, I first collapse it to book-level metrics before comparing by gender.

In [231]:
merged_books = df_merged.copy()
merged_books["nyt_match"] = merged_books["nyt_title"].notna()

book_bestseller_metrics = (
    merged_books.groupby(["author", "title"], as_index=False)
    .agg(
        is_bestseller=("nyt_match", "any"),
        weeks_on_list=("nyt_match", "sum"),
    )
)

df_with_bestseller = df.merge(book_bestseller_metrics, how="left", on=["author", "title"])
df_with_bestseller["is_bestseller"] = df_with_bestseller["is_bestseller"].fillna(False).astype(bool)
df_with_bestseller["weeks_on_list"] = df_with_bestseller["weeks_on_list"].fillna(0).astype(int)

bestseller_by_gender = (
    df_with_bestseller.groupby("author_gender", as_index=False)
    .agg(
        share_bestseller=("is_bestseller", "mean"),
        average_weeks_on_list=("weeks_on_list", "mean"),
        books=("title", "count"),
    )
)

bestseller_by_decade_gender = (
    df_with_bestseller.groupby(["decade", "author_gender"], as_index=False)
    .agg(
        share_bestseller=("is_bestseller", "mean"),
        books=("title", "count"),
    )
)

display(book_bestseller_metrics.head())
display(bestseller_by_gender)


,author,title,is_bestseller,weeks_on_list
0,Agatha Christie,And Then There Were None,False,0
1,Agatha Christie,Murder on the Orient Express,False,0
2,Agatha Christie,The Murder of Roger Ackroyd,False,0
3,Alan Paton,"Cry, The Beloved Country",False,0
4,Albert Camus,The Fall,False,0


,author_gender,share_bestseller,average_weeks_on_list,books
0,female,0.020690,0.558621,145
1,male,0.019718,0.422535,355


The first datafram shows the new row *is_bestseller* and *weeks_on_list* columns, which help lead the analysis on the merged dataset statistics. They will measure success in both areas of the book world.

The second dataframe is showcasing the averages and counts for the different statistics surrounding male and female novels. Just looking at the data, it seems like women have better shares of bestsellers and average weeks on the list.

### Share of Top 500 Books That Also Became NYT Bestsellers

This chart shows the percentage of Top 500 books by each gender group that appear in the NYT bestseller data at least once. This is a book-level comparison, so a book with many weekly NYT rows still counts as one bestseller for the share calculation.

In [232]:
bestseller_share_chart = (
    alt.Chart(bestseller_by_gender)
    .mark_bar()
    .encode(
        x=alt.X("author_gender:N", title="Author gender"),
        y=alt.Y("share_bestseller:Q", title="Share that appeared on NYT list", axis=alt.Axis(format="%")),
        color=alt.Color("author_gender:N", title="Author gender"),
        tooltip=[
            alt.Tooltip("author_gender:N", title="Author gender"),
            alt.Tooltip("share_bestseller:Q", title="Share", format=".1%"),
            alt.Tooltip("books:Q", title="Top 500 books"),
        ],
    )
    .properties(
        title="Share of Top 500 Books with NYT Bestseller Appearance by Gender",
        width=520,
        height=340,
    )
)

bestseller_share_chart


alt.Chart(...)

This chart helps separate two ideas that can otherwise get mixed together: being included in a canon-oriented Top 500 list and appearing in a weekly bestseller list. A higher percentage here means more books in that gender group crossed into NYT bestseller visibility, not that they had better Top 500 rankings.

What we can see specifically from this graph is that the women's books are more popular, or did better on average than the mens books. Even though there were less books than the women, the man authored books still had more books NOT reach the NYT bestsellers. 

#### Average Weeks on the NYT List by Gender

Weeks on list measures bestseller longevity. It is different from the picked ranking because it reflects repeated weekly appearances in the NYT data rather than position in the Top 500 dataset.

In [233]:
weeks_chart = (
    alt.Chart(bestseller_by_gender)
    .mark_bar()
    .encode(
        x=alt.X("author_gender:N", title="Author gender"),
        y=alt.Y("average_weeks_on_list:Q", title="Average NYT weekly rows per Top 500 book"),
        color=alt.Color("author_gender:N", title="Author gender"),
        tooltip=[
            alt.Tooltip("author_gender:N", title="Author gender"),
            alt.Tooltip("average_weeks_on_list:Q", title="Average weeks", format=".2f"),
            alt.Tooltip("books:Q", title="Top 500 books"),
        ],
    )
    .properties(
        title="Average NYT Bestseller Longevity by Author Gender",
        width=520,
        height=340,
    )
)

weeks_chart


alt.Chart(...)

A larger value here suggests that books by that gender group stayed visible in the NYT weekly data for longer on average. This should be read as a popularity or market-history measure, not as a measure of literary quality.

This graph ultimately shows that women's novels were more popular than men's novels. By a larger margin than the first graph as well.

#### Bestseller Presence Over Time by Gender

This decade chart adds time back into the NYT extension. It asks whether Top 500 books from some decades were more likely to also appear in the NYT bestseller data, and whether that pattern differs by author gender.

In [234]:
bestseller_decade_chart = (
    alt.Chart(bestseller_by_decade_gender)
    .mark_line(point=True)
    .encode(
        x=alt.X("decade:O", title="Publication decade"),
        y=alt.Y("share_bestseller:Q", title="Share with NYT appearance", axis=alt.Axis(format="%")),
        color=alt.Color("author_gender:N", title="Author gender"),
        tooltip=[
            alt.Tooltip("decade:O", title="Decade"),
            alt.Tooltip("author_gender:N", title="Author gender"),
            alt.Tooltip("share_bestseller:Q", title="Share", format=".1%"),
            alt.Tooltip("books:Q", title="Top 500 books"),
        ],
    )
    .properties(
        title="NYT Bestseller Presence Among Top 500 Books by Decade and Gender",
        width=700,
        height=380,
    )
)

bestseller_decade_chart


alt.Chart(...)

This plot shows that this list didn't start until around 1910, well after a lot of the ranked books were written. There are moments where men do better than women and vice versa, but on average the women are holding more of the appearances on the NYT bestselling than the men are. Time doesn't seem to have a ton of effect on the graph itself, besides for the fact that it continuously has bigger peaks as time goes on. There is a limited number of books for this data, which may not allow for the best assumptions to be made.

### Conclusion

Through the EDA that I conducted, I was able to find some insights about the data. 

- Women weren't represented well historically, and it wasn't until mid 1800s that they started making more waves in the novel/author scene
- As time went on, more authors started getting selected for the top 500 list, but the percentage of women from those groups grew as well
- AS time went on, the average rank of the newer books was worse than the average rank of the older books
- Novels written by women were found to be more popular, as seen on the NYT bestsellers list. Not only did more women make the list, but they stayed on for a longer time on average then men did.

These results ould have confounding variables, like the era the books are from. The size difference between the men and women authored novels in the dataset to begin with could also cause a difference. 

